In [20]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Colab Data/WELFake_News_Data.csv')

In [21]:
df

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
2,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
3,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1
4,5,About Time! Christian Group Sues Amazon and SP...,All we can say on this one is it s about time ...,1
...,...,...,...,...
62195,72127,WIKILEAKS EMAIL SHOWS CLINTON FOUNDATION FUNDS...,An email released by WikiLeaks on Sunday appea...,1
62196,72129,Russians steal research on Trump in hack of U....,WASHINGTON (Reuters) - Hackers believed to be ...,0
62197,72130,WATCH: Giuliani Demands That Democrats Apolog...,"You know, because in fantasyland Republicans n...",1
62198,72131,Migrants Refuse To Leave Train At Refugee Camp...,Migrants Refuse To Leave Train At Refugee Camp...,0


In [22]:
from sklearn.model_selection import train_test_split
train_set, temp_set = train_test_split(df, test_size=0.2, random_state=42)

In [23]:
val_set, test_set = train_test_split(temp_set, test_size=0.5, random_state=42)

In [24]:
from datasets import Dataset
train_dataset = Dataset.from_pandas(train_set.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_set.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_set.reset_index(drop=True))

In [25]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tokenizer_fn(example):
  return tokenizer(example['text'], truncation=True, padding="max_length", max_length=512)
train_dataset = train_dataset.map(tokenizer_fn, batched=True)
val_dataset = val_dataset.map(tokenizer_fn, batched=True)
test_dataset = test_dataset.map(tokenizer_fn, batched=True)


Map:   0%|          | 0/49760 [00:00<?, ? examples/s]

Map:   0%|          | 0/6220 [00:00<?, ? examples/s]

Map:   0%|          | 0/6220 [00:00<?, ? examples/s]

In [26]:
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [27]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

In [28]:
def compute_metrics(eval_pred):
  logits, lables = eval_pred
  predictions = np.argmax(logits, axis=-1)
  return {
      "accuracy": accuracy_score(lables, predictions),
      "f1": f1_score(lables, predictions)
  }

In [29]:
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='epoch',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()



[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.098867,0.027385,0.992926,0.992006
2,0.000156,0.027159,0.994534,0.993809


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=12440, training_loss=0.04985670432136948, metrics={'train_runtime': 1607.3408, 'train_samples_per_second': 61.916, 'train_steps_per_second': 7.739, 'total_flos': 1.318315551424512e+16, 'train_loss': 0.04985670432136948, 'epoch': 2.0})

In [30]:
model.save_pretrained('/content/drive/MyDrive/Colab Data/distilbert-fakenews-final')
tokenizer.save_pretrained('/content/drive/MyDrive/Colab Data/distilbert-fakenews-final')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Colab Data/distilbert-fakenews-final/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Data/distilbert-fakenews-final/tokenizer.json')

In [33]:
results = trainer.evaluate(test_dataset)
print(results)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000156,0.047540,2,0.991158,0.989962


{'eval_loss': 0.04753972962498665, 'eval_accuracy': 0.9911575562700965, 'eval_f1': 0.9899616718379266}


In [35]:
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

from sklearn.metrics import confusion_matrix, classification_report
print(confusion_matrix(labels, preds))
print(classification_report(labels, preds, target_names=["real", "fake"]))

[[3453   28]
 [  27 2712]]
              precision    recall  f1-score   support

        real       0.99      0.99      0.99      3481
        fake       0.99      0.99      0.99      2739

    accuracy                           0.99      6220
   macro avg       0.99      0.99      0.99      6220
weighted avg       0.99      0.99      0.99      6220

